In [ ]:
from fast_plate_ocr.inference.hub import download_model

# This downloads the ONNX file and config, returning their local paths
model_path, config_path = download_model("cct-s-v2-global-model")

print(f"ONNX Model saved at: {model_path}")
print(f"Config File saved at: {config_path}")


In [2]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
import yaml

BASE_DIR = Path("surveillance-app/data/dataset/vehicle_detection")
if not BASE_DIR.exists():
    # Fallback if working inside root or different subpath
    BASE_DIR = Path("surveillance-app/data/dataset/vehicle_detection")

print(f"Inspecting datasets inside: {BASE_DIR.resolve()}\n")

datasets = [
    "IDD_Detection",
    "IDD_FGVD",
    "Indian emergency vehicles",
    "Indian Vehicle & plate",
    "indian-emergency-vehicles-dataset",
    "vehicle cam dataset",
    "vehicle",
    "Indian Vehicle Classification"
]

summary = {}

for ds_name in datasets:
    ds_path = BASE_DIR / ds_name
    if not ds_path.exists():
        print(f"[-] Directory not found: {ds_path}")
        continue
    
    print(f"==================================================")
    print(f"Scanning: {ds_name}")
    print(f"==================================================")
    
    dataset_info = {
        "format": "Unknown",
        "yaml_classes": [],
        "xml_or_txt_classes": Counter(),
        "total_annotations": 0
    }
    
    # 1. Check for data.yaml / dataset.yaml
    yaml_files = list(ds_path.glob("*.yaml")) + list(ds_path.glob("*.yml"))
    if yaml_files:
        dataset_info["format"] = "YOLO (Roboflow)"
        try:
            with open(yaml_files[0], 'r') as f:
                data_cfg = yaml.safe_load(f)
                names = data_cfg.get("names", [])
                dataset_info["yaml_classes"] = names
                print(f"  [YAML Found]: {yaml_files[0].name}")
                print(f"  [YAML Classes]: {names}")
        except Exception as e:
            print(f"  [!] Failed reading YAML: {e}")
            
    # 2. Check for Pascal VOC XMLs (like IDD_Detection)
    xml_files = list(ds_path.glob("**/*.xml"))
    if xml_files:
        dataset_info["format"] = "Pascal VOC (XML)"
        print(f"  [XML Annotations Found]: {len(xml_files)} files. Sampling classes...")
        xml_counter = Counter()
        # Sample up to 1500 XMLs for speed or scan all
        for xml_p in xml_files[:2000]:
            try:
                tree = ET.parse(xml_p)
                for obj in tree.getroot().findall("object"):
                    name = obj.find("name")
                    if name is not None and name.text:
                        xml_counter[name.text.strip().lower()] += 1
            except Exception:
                continue
        dataset_info["xml_or_txt_classes"] = xml_counter
        dataset_info["total_annotations"] = len(xml_files)
        print(f"  [Top XML Classes Sampled]: {dict(xml_counter.most_common(15))}")

    # 3. Check for raw YOLO txt files if no YAML is present or in subfolders
    if not yaml_files and not xml_files:
        txt_files = list(ds_path.glob("**/*.txt"))
        # Exclude readme / license files
        txt_labels = [p for p in txt_files if not p.name.startswith("README") and p.name not in ["train.txt", "val.txt", "test.txt"]]
        if txt_labels:
            dataset_info["format"] = "YOLO TXT (Raw)"
            print(f"  [Raw TXT Labels Found]: {len(txt_labels)} label files")
            # Count raw class IDs
            id_counter = Counter()
            for lp in txt_labels[:1000]:
                try:
                    with open(lp, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if parts:
                                id_counter[parts[0]] += 1
                except Exception:
                    continue
            print(f"  [Class IDs Found in TXT]: {dict(id_counter.most_common())}")
            
    summary[ds_name] = dataset_info
    print()

print("\n" + "="*80)
print("SCAN COMPLETED")
print("="*80)

Inspecting datasets inside: /Users/aseem/Project/surveillance-app/data/dataset/vehicle_detection

Scanning: IDD_Detection
  [XML Annotations Found]: 41857 files. Sampling classes...
  [Top XML Classes Sampled]: {'car': 4160, 'rider': 3170, 'motorcycle': 3137, 'person': 1494, 'autorickshaw': 976, 'bus': 859, 'truck': 651, 'vehicle fallback': 374, 'traffic sign': 159, 'animal': 149, 'bicycle': 112, 'traffic light': 9, 'trailer': 1}

Scanning: IDD_FGVD
  [XML Annotations Found]: 5502 files. Sampling classes...
  [Top XML Classes Sampled]: {'scooter_honda_activa': 1029, 'autorickshaw_others': 899, 'bus': 447, 'autorickshaw_bajaj': 359, 'motorcycle_hero_splendor': 284, 'car_marutisuzuki_dzire': 279, 'car_tatamotors_indica': 276, 'motorcycle_bajaj_pulsar150': 234, 'car_others': 228, 'motorcycle_others': 209, 'truck_tata': 197, 'car_toyota_innova': 179, 'car_toyota_etios': 164, 'motorcycle_honda_shine': 159, 'scooter_suzuki_access': 148}

Scanning: Indian emergency vehicles
  [YAML Found]: da

In [2]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil
import random
import yaml
from collections import defaultdict
from tqdm import tqdm

candidate_paths = [
    Path("data/dataset/vehicle_detection"),
    Path("surveillance-app/data/dataset/vehicle_detection"),
    Path(".")
]
BASE_DIR = next((p for p in candidate_paths if (p / "IDD_Detection").exists()), None)
assert BASE_DIR is not None, "Could not find vehicle_detection dataset folder."

OUT_DIR = BASE_DIR / "unified_vehicle_detection_5class"

CANONICAL_CLASSES = [
    "car",           # 0
    "two_wheeler",   # 1
    "auto_rickshaw", # 2
    "bus",           # 3
    "truck"          # 4
]

CLASS_TO_ID = {name: i for i, name in enumerate(CANONICAL_CLASSES)}

def map_raw_label(raw: str):
    label = raw.strip().lower()

    # Drop noise, signs, text, sub-assemblies, animals, pedestrians, and ambiguous fallbacks
    if any(ign in label for ign in [
        "fallback", "lamp", "text", "sign", "ladder", "symbol", "writing",
        "plate", "person", "pedestrian", "animal", "horse", "dog", "cow",
        "traffic light", "traffic sign", "road_sign"
    ]):
        return None

    if label in ["vehicle", "army"]:
        return None

    # Auto & E-Rickshaws -> Class 2
    if any(k in label for k in ["auto rickshaw", "autorickshaw", "e-rickshaw", "rikshaw"]):
        return 2
    if label == "auto":
        return 2

    # Two-wheelers -> Class 1 (or 101 token for rider fusion)
    if any(k in label for k in ["motorcycle", "scooty", "scooter", "bike", "motobike", "bicycle"]):
        return 1
    if label == "rider":
        return 101

    # Buses -> Class 3
    if "bus" in label:
        return 3

    # Trucks, Dumpers, Tempos, Mini-Loaders -> Class 4
    if any(k in label for k in [
        "truck", "dumper", "tempo traveller", "mini loading", "trailer", "fire_truck", "lorry"
    ]):
        return 4

    # Cars -> Class 0
    if any(k in label for k in ["car", "jeep", "suv", "sedan", "hatchback", "taxi", "van", "ambulance", "police"]):
        return 0

    return None

def find_label_file(img_p: Path) -> Path:
    """Finds label file whether co-located or in a separate sibling labels/ directory."""
    # 1. Co-located (e.g. images/001.jpg -> images/001.txt)
    cand1 = img_p.with_suffix(".txt")
    if cand1.exists():
        return cand1

    # 2. Path replacement: replace any 'images' folder with 'labels'
    parts = list(img_p.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() == "images":
            parts[i] = "labels"
            cand2 = Path(*parts).with_suffix(".txt")
            if cand2.exists():
                return cand2
            break

    # 3. Direct parent replacement
    parent = img_p.parent
    if parent.name.lower() == "images":
        cand3 = parent.parent / "labels" / f"{img_p.stem}.txt"
        if cand3.exists():
            return cand3

    return None

def merge_overlapping_riders(boxes, iou_thresh=0.10):
    vehicles = [b for b in boxes if b[0] != 101]
    riders = [b for b in boxes if b[0] == 101]

    merged_vehicles = []
    used_riders = set()

    for cls_id, vx1, vy1, vx2, vy2 in vehicles:
        if cls_id == 1:
            cur_x1, cur_y1, cur_x2, cur_y2 = vx1, vy1, vx2, vy2
            v_area = (vx2 - vx1) * (vy2 - vy1)

            for r_idx, (_, rx1, ry1, rx2, ry2) in enumerate(riders):
                if r_idx in used_riders:
                    continue

                ix1 = max(cur_x1, rx1)
                iy1 = max(cur_y1, ry1)
                ix2 = min(cur_x2, rx2)
                iy2 = min(cur_y2, ry2)
                iw = max(0.0, ix2 - ix1)
                ih = max(0.0, iy2 - iy1)
                inter = iw * ih

                if inter > 0:
                    r_area = (rx2 - rx1) * (ry2 - ry1)
                    union = v_area + r_area - inter
                    iou = inter / union if union > 0 else 0
                    if iou > iou_thresh or (ix1 >= rx1 and ix2 <= rx2):
                        cur_x1 = min(cur_x1, rx1)
                        cur_y1 = min(cur_y1, ry1)
                        cur_x2 = max(cur_x2, rx2)
                        cur_y2 = max(cur_y2, ry2)
                        used_riders.add(r_idx)

            merged_vehicles.append((1, cur_x1, cur_y1, cur_x2, cur_y2))
        else:
            merged_vehicles.append((cls_id, vx1, vy1, vx2, vy2))

    for r_idx, (_, rx1, ry1, rx2, ry2) in enumerate(riders):
        if r_idx not in used_riders:
            merged_vehicles.append((1, rx1, ry1, rx2, ry2))

    return merged_vehicles

# Reset output directories
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

for split in ["train", "val"]:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

class_counts = defaultdict(int)
total_images_processed = 0

# --- 1. Process Pascal VOC XMLs (IDD) ---
def process_xml_dataset(ds_path: Path):
    global total_images_processed
    if not ds_path.exists():
        return

    print(f"\nProcessing Pascal VOC: {ds_path.name}")
    img_map = {}
    valid_exts = {".jpg", ".jpeg", ".png"}
    for p in ds_path.glob("**/*"):
        if p.is_file() and p.suffix.lower() in valid_exts and "Annotations" not in str(p):
            img_map[p.stem] = p

    xml_files = list(ds_path.glob("**/*.xml"))
    random.seed(42)

    for xml_p in tqdm(xml_files, desc=f"Converting {ds_path.name}"):
        stem = xml_p.stem
        img_path = img_map.get(stem)
        if not img_path:
            continue

        try:
            tree = ET.parse(xml_p)
            root = tree.getroot()
            size = root.find("size")
            if size is None:
                continue
            w = float(size.find("width").text)
            h = float(size.find("height").text)
            if w <= 0 or h <= 0:
                continue

            raw_boxes = []
            for obj in root.findall("object"):
                name_elem = obj.find("name")
                if name_elem is None or not name_elem.text:
                    continue
                cls_id = map_raw_label(name_elem.text)
                if cls_id is None:
                    continue

                b = obj.find("bndbox")
                x1 = max(0.0, float(b.find("xmin").text))
                y1 = max(0.0, float(b.find("ymin").text))
                x2 = min(w, float(b.find("xmax").text))
                y2 = min(h, float(b.find("ymax").text))

                # Discard noise/horizon boxes smaller than 24px
                if (x2 - x1) >= 24 and (y2 - y1) >= 24:
                    raw_boxes.append((cls_id, x1, y1, x2, y2))

            if not raw_boxes:
                continue

            consolidated = merge_overlapping_riders(raw_boxes)

            yolo_lines = []
            for cls_id, x1, y1, x2, y2 in consolidated:
                bw = x2 - x1
                bh = y2 - y1
                cx = (x1 + bw / 2.0) / w
                cy = (y1 + bh / 2.0) / h
                nw = bw / w
                nh = bh / h
                yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                class_counts[CANONICAL_CLASSES[cls_id]] += 1

            if not yolo_lines:
                continue

            split = "val" if random.random() < 0.15 else "train"
            prefix = f"{ds_path.name.replace(' ', '_')}_{stem}"
            out_img = OUT_DIR / "images" / split / f"{prefix}{img_path.suffix.lower()}"
            out_lbl = OUT_DIR / "labels" / split / f"{prefix}.txt"

            shutil.copy2(img_path, out_img)
            with open(out_lbl, "w") as f:
                f.write("\n".join(yolo_lines) + "\n")

            total_images_processed += 1
        except Exception:
            continue

# --- 2. Process YOLO TXT Datasets ---
def process_yolo_dataset(ds_path: Path):
    global total_images_processed
    if not ds_path.exists():
        return

    print(f"\nProcessing YOLO TXT: {ds_path.name}")
    yaml_files = list(ds_path.glob("**/*.yaml")) + list(ds_path.glob("**/*.yml"))
    if not yaml_files:
        print(f"[-] No yaml found in {ds_path}, skipping.")
        return

    with open(yaml_files[0], "r") as f:
        cfg = yaml.safe_load(f)

    raw_names = cfg.get("names", [])
    if isinstance(raw_names, dict):
        raw_names = [raw_names[i] for i in sorted(raw_names.keys())]

    idx_map = {}
    for i, raw_name in enumerate(raw_names):
        mapped = map_raw_label(raw_name)
        if mapped is not None and mapped != 101:
            idx_map[i] = mapped

    print(f"Mapped {len(idx_map)}/{len(raw_names)} classes for {ds_path.name}: {idx_map}")

    img_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        img_files.extend(ds_path.glob(f"**/{ext}"))

    # Skip files located in output directory if scanning root
    img_files = [p for p in img_files if "unified_vehicle_detection" not in str(p)]
    random.seed(42)

    copied_for_ds = 0
    for img_p in tqdm(img_files, desc=f"Converting {ds_path.name}"):
        lbl_p = find_label_file(img_p)
        if not lbl_p:
            continue

        yolo_lines = []
        try:
            with open(lbl_p, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        raw_id = int(parts[0])
                        if raw_id in idx_map:
                            cls_id = idx_map[raw_id]
                            cx, cy, nw, nh = map(float, parts[1:5])
                            # Filter small horizon noise (< 0.035 normalized ~ 22px at 640)
                            if nw >= 0.035 and nh >= 0.035:
                                yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                                class_counts[CANONICAL_CLASSES[cls_id]] += 1

            if not yolo_lines:
                continue

            p_str = str(img_p).lower()
            split = "val" if ("valid" in p_str or "val" in p_str) else "train"

            prefix = f"{ds_path.name.replace(' ', '_')}_{img_p.stem}"
            out_img = OUT_DIR / "images" / split / f"{prefix}{img_p.suffix.lower()}"
            out_lbl = OUT_DIR / "labels" / split / f"{prefix}.txt"

            shutil.copy2(img_p, out_img)
            with open(out_lbl, "w") as f:
                f.write("\n".join(yolo_lines) + "\n")

            total_images_processed += 1
            copied_for_ds += 1
        except Exception:
            continue

    print(f"Successfully converted and copied {copied_for_ds} images from {ds_path.name}")

# Run across all datasets
xml_datasets = ["IDD_Detection", "IDD_FGVD"]
yolo_datasets = [
    "Indian emergency vehicles",
    "Indian Vehicle & plate",
    "indian-emergency-vehicles-dataset",
    "vehicle cam dataset",
    "vehicle",
    "Indian Vehicle Classification"
]

for d in xml_datasets:
    process_xml_dataset(BASE_DIR / d)

for d in yolo_datasets:
    process_yolo_dataset(BASE_DIR / d)

# Write unified data.yaml
data_yaml_dict = {
    "path": str(OUT_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CANONICAL_CLASSES)}
}

with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.dump(data_yaml_dict, f, sort_keys=False)

print("\n" + "=" * 80)
print(f"SUCCESS: Unified Clean 5-Class Dataset generated at:\n  {OUT_DIR.resolve()}")
print(f"Total Merged Images: {total_images_processed}")
print("Class Instances Summary:")
for k, v in class_counts.items():
    print(f"  - {k:<20}: {v:>8} instances")
print("=" * 80)


Processing Pascal VOC: IDD_Detection


Converting IDD_Detection: 100%|██████████| 41857/41857 [00:46<00:00, 895.86it/s] 



Processing Pascal VOC: IDD_FGVD


Converting IDD_FGVD: 100%|██████████| 5502/5502 [00:14<00:00, 387.87it/s]



Processing YOLO TXT: Indian emergency vehicles
Mapped 11/24 classes for Indian emergency vehicles: {2: 0, 3: 0, 4: 0, 7: 2, 8: 1, 9: 3, 10: 0, 11: 4, 17: 0, 21: 4, 22: 4}


Converting Indian emergency vehicles: 100%|██████████| 19959/19959 [00:38<00:00, 516.12it/s] 


... (5 outputs left out)

Converting vehicle: 100%|██████████| 15452/15452 [00:28<00:00, 547.45it/s] 


Successfully converted and copied 12741 images from vehicle

Processing YOLO TXT: Indian Vehicle Classification
Mapped 16/17 classes for Indian Vehicle Classification: {0: 0, 1: 2, 2: 3, 3: 3, 4: 0, 5: 4, 6: 4, 7: 4, 8: 2, 9: 4, 11: 1, 12: 4, 13: 4, 14: 4, 15: 4, 16: 4}


Converting Indian Vehicle Classification: 100%|██████████| 9175/9175 [00:17<00:00, 518.53it/s] 

Successfully converted and copied 7407 images from Indian Vehicle Classification

SUCCESS: Unified Clean 5-Class Dataset generated at:
  /Users/aseem/Project/surveillance-app/data/dataset/vehicle_detection/unified_vehicle_detection_5class
Total Merged Images: 133974
Class Instances Summary:
  - two_wheeler         :   161424 instances
  - bus                 :    25600 instances
  - car                 :   151651 instances
  - truck               :    49777 instances
  - auto_rickshaw       :    60752 instances


In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = Path("data/dataset/vehicle_detection/unified_vehicle_detection_5class/data.yaml")
if not DATA_YAML.exists():
    DATA_YAML = Path("surveillance-app/data/dataset/vehicle_detection/unified_vehicle_detection_5class/data.yaml")

device = "mps" if torch.backends.mps.is_available() else ("0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Start from baseline COCO weights
model = YOLO("yolo26n.pt")

results = model.train(
    data=str(DATA_YAML.resolve()),
    epochs=30,             # 134k images is huge; 20-25 epochs is plenty for convergence
    patience=8,
    imgsz=640,
    batch=32 if device == "0" else 16,
    workers=8,
    device=device,
    freeze=10,             # FREEZE BACKBONE: locks early edge/shape layers to prevent forgetting
    optimizer="AdamW",
    lr0=0.0005,            # Lower learning rate so it fine-tunes without blowing up weights
    lrf=0.01,
    warmup_epochs=1.5,
    mosaic=0.5,            # Lower mosaic so elevated camera vehicles retain natural contours
    close_mosaic=5,
    box=7.5,
    cls=0.5,
    project="runs/vehicle_detector",
    name="yolo26n_surveillance_134k",
    save=True,
    exist_ok=True
)

Training on device: mps
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.198 🚀 Python-3.12.13 torch-2.12.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/aseem/Project/surveillance-app/data/dataset/vehicle_detection/unified_vehicle_detection_5class/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.

KeyboardInterrupt: 

In [4]:
from ultralytics import YOLO
import shutil

best_pt = "/Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.pt"
model = YOLO(best_pt)
exported_onnx = model.export(format="onnx", imgsz=640, simplify=True, opset=17)

# Copy to rust models directory
shutil.copy2(exported_onnx, "/Users/aseem/Project/surveillance-app/data/models/yolo26n.onnx")
print("Exported to Rust data/models/yolo26n.onnx!")

Ultralytics 8.4.198 🚀 Python-3.12.13 torch-2.12.0 CPU (Apple M4 Max)
YOLO26n summary (fused): 121 layers, 2,390,707 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from '/Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.2 MB)

ONNX: starting export with onnx 1.17.0 opset 17...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 0.6s, saved as '/Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.onnx' (9.4 MB)

Export complete (0.6s)
Results saved to /Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.onnx
Predict:         yolo predict task=detect model=/Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/Users/aseem/Project/runs/detect/runs/vehicle_detector/yolo26n_indi

In [3]:
from ultralytics import YOLO
import torch

model = YOLO("runs/detect/runs/vehicle_detector/yolo26n_indian_5class_v2/weights/best.pt")

for img_name in ["1.jpg", "2.jpg", "3.jpg"]:
    img_path = f"surveillance-app/data/tests/{img_name}"
    results = model.predict(img_path, conf=0.15)
    print(f"\n================ Raw Predictions for {img_name} ================")
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = model.names[cls_id]
            print(f"Detected: {name:<15} | Conf: {conf*100:.1f}% | Box: {box.xyxy[0].tolist()}")


image 1/1 /Users/aseem/Project/surveillance-app/data/tests/1.jpg: 640x480 1 car, 18.3ms
Speed: 1.4ms preprocess, 18.3ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)

================ Raw Predictions for 1.jpg ================
Detected: car             | Conf: 87.7% | Box: [117.61676788330078, 387.04595947265625, 517.6699829101562, 642.5558471679688]

image 1/1 /Users/aseem/Project/surveillance-app/data/tests/2.jpg: 448x640 4 cars, 1 bus, 17.6ms
Speed: 0.7ms preprocess, 17.6ms inference, 0.2ms postprocess per image at shape (1, 3, 448, 640)

================ Raw Predictions for 2.jpg ================
Detected: car             | Conf: 81.0% | Box: [258.89910888671875, 108.1072006225586, 767.68017578125, 564.9108276367188]
Detected: car             | Conf: 64.2% | Box: [239.1068572998047, 224.12440490722656, 305.3428039550781, 275.9938049316406]
Detected: car             | Conf: 38.3% | Box: [721.4906616210938, 196.52980041503906, 795.0410766601562, 251.8914337158203]

In [1]:
import shutil
from pathlib import Path
from ultralytics import YOLO

# 1. Load the pre-trained standard YOLO26 nano weights
model = YOLO("yolo26n.pt")

# 2. Export with static batch/dims (opset 17 with onnxslim optimization)
exported_path = model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17
)

# 3. Destination target
target_path = Path("/Users/aseem/Project/surveillance-app/data/models/yolo26n.onnx")
target_path.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(exported_path, target_path)
print(f"Model successfully saved to: {target_path.resolve()}")

Ultralytics 8.4.198 🚀 Python-3.12.13 torch-2.12.0 CPU (Apple M4 Max)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

ONNX: starting export with onnx 1.17.0 opset 17...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 0.8s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (0.9s)
Results saved to /Users/aseem/Project/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640  
Visualize:       https://netron.app
Model successfully saved to: /Users/aseem/Project/surveillance-app/data/models/yolo26n.onnx
